In [45]:
import pandas as pd
from sqlalchemy import create_engine, select, func,text
from sqlalchemy.orm import Session
import os
import sys

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../..")))

In [ ]:
from src.database.entidades import (motor,Titulo,Genero,Titulo_Genero,Puntaje,Persona,
                                    Profesion,Profesion_Titulo,Titulo_Alternativo)

session = Session(motor)

Todos los títulos

In [25]:
titulos = session.execute(select(Titulo)).scalars().all()
df_titulos = pd.DataFrame([{
    "id": t.id,
    "tipo": t.tipo.value,
    "titulo": t.titulo,
    "fecha_estreno": t.fecha_estreno,
    "duracion": t.duracion
} for t in titulos])
display(df_titulos.head())

2025-11-02 22:37:51,540 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-11-02 22:37:51,543 INFO sqlalchemy.engine.Engine SELECT titulo.id, titulo.tipo, titulo.titulo, titulo.duracion, titulo.sinopsis, titulo.poster, titulo.fecha_estreno 
FROM titulo
2025-11-02 22:37:51,544 INFO sqlalchemy.engine.Engine [cached since 1711s ago] ()


,id,tipo,titulo,fecha_estreno,duracion
0,tt0116991,movie,Mariette in Ecstasy,2019-01-01,101
1,tt0385887,movie,Motherless Brooklyn,2019-01-01,144
2,tt0437086,movie,Alita: Battle Angel,2019-01-01,122
3,tt0441881,movie,Danger Close,2019-01-01,118
4,tt0448115,movie,Shazam!,2019-01-01,132


Filtrar películas por tipo

In [30]:
peliculas = session.execute(
    select(Titulo).where(Titulo.tipo == 'PELICULA')
).scalars().all()

print(f"🎞️ Total películas: {len(peliculas)}")
[p.titulo for p in peliculas[:10]]

2025-11-02 22:39:33,819 INFO sqlalchemy.engine.Engine SELECT titulo.id, titulo.tipo, titulo.titulo, titulo.duracion, titulo.sinopsis, titulo.poster, titulo.fecha_estreno 
FROM titulo 
WHERE titulo.tipo = ?
2025-11-02 22:39:33,821 INFO sqlalchemy.engine.Engine [cached since 736.7s ago] ('PELICULA',)
🎞️ Total películas: 10052


['Mariette in Ecstasy',
 'Motherless Brooklyn',
 'Alita: Battle Angel',
 'Danger Close',
 'Shazam!',
 'The Last Full Measure',
 'The Dirt',
 'Dirt Music',
 'Pet Sematary',
 'Bolden']

In [27]:
peliculas = session.execute(
    select(Titulo).where(Titulo.tipo == 'TV_MOVIE')
).scalars().all()

print(f"🎞️ Total películas: {len(peliculas)}")
[p.titulo for p in peliculas[:10]]

2025-11-02 22:38:35,720 INFO sqlalchemy.engine.Engine SELECT titulo.id, titulo.tipo, titulo.titulo, titulo.duracion, titulo.sinopsis, titulo.poster, titulo.fecha_estreno 
FROM titulo 
WHERE titulo.tipo = ?
2025-11-02 22:38:35,722 INFO sqlalchemy.engine.Engine [cached since 678.6s ago] ('TV_MOVIE',)
🎞️ Total películas: 1202


["L'enigma Verdaguer",
 'Past Never Dies',
 'Kilimanjaro: The Bigger Red Nose Climb',
 'Patsy & Loretta',
 'Murtomaa',
 'Raid',
 'The Dating List',
 'The Boiling Water LAMA',
 'The Wrong Stepmother',
 'Gegen die Angst']

Mostrar géneros de un título

In [31]:
titulo_id = 'tt10004372'
generos = session.execute(
    select(Genero.nombre)
    .join(Titulo_Genero)
    .where(Titulo_Genero.id_titulo == titulo_id)
).scalars().all()

print(f"Géneros asociados a {titulo_id}: {generos}")

2025-11-02 22:41:38,619 INFO sqlalchemy.engine.Engine SELECT genero.nombre 
FROM genero JOIN titulo_genero ON genero.id = titulo_genero.id_genero 
WHERE titulo_genero.id_titulo = ?
2025-11-02 22:41:38,620 INFO sqlalchemy.engine.Engine [cached since 831.7s ago] ('tt10004372',)
Géneros asociados a tt10004372: ['Documentary']


In [32]:
titulo_id = 'tt0385887'
generos = session.execute(
    select(Genero.nombre)
    .join(Titulo_Genero)
    .where(Titulo_Genero.id_titulo == titulo_id)
).scalars().all()

print(f"Géneros asociados a {titulo_id}: {generos}")

2025-11-02 22:41:49,574 INFO sqlalchemy.engine.Engine SELECT genero.nombre 
FROM genero JOIN titulo_genero ON genero.id = titulo_genero.id_genero 
WHERE titulo_genero.id_titulo = ?
2025-11-02 22:41:49,577 INFO sqlalchemy.engine.Engine [cached since 842.7s ago] ('tt0385887',)
Géneros asociados a tt0385887: ['Crime', 'Drama', 'Mystery']


Puntaje de un título

In [33]:
puntaje = session.execute(
    select(Puntaje.promedio, Puntaje.cantidad_votos)
    .where(Puntaje.id_titulo == titulo_id)
).first()

puntaje if puntaje else "Sin puntaje"

2025-11-02 22:42:52,503 INFO sqlalchemy.engine.Engine SELECT puntaje.promedio, puntaje.cantidad_votos 
FROM puntaje 
WHERE puntaje.id_titulo = ?
2025-11-02 22:42:52,505 INFO sqlalchemy.engine.Engine [generated in 0.00192s] ('tt0385887',)


(6.8, 65258)

Personas que participaron en un título

In [34]:
personas = session.execute(
    select(Persona.nombre, Profesion.nombre, Profesion_Titulo.nombre_personaje)
    .join(Profesion_Titulo, Persona.id == Profesion_Titulo.id_persona)
    .join(Profesion, Profesion.id == Profesion_Titulo.id_profesion)
    .where(Profesion_Titulo.id_titulo == titulo_id)
).all()

pd.DataFrame(personas, columns=["Persona", "Profesión", "Personaje"])

2025-11-02 22:43:36,650 INFO sqlalchemy.engine.Engine SELECT persona.nombre, profesion.nombre AS nombre_1, profesion_titulo.nombre_personaje 
FROM persona JOIN profesion_titulo ON persona.id = profesion_titulo.id_persona JOIN profesion ON profesion.id = profesion_titulo.id_profesion 
WHERE profesion_titulo.id_titulo = ?
2025-11-02 22:43:36,652 INFO sqlalchemy.engine.Engine [generated in 0.00183s] ('tt0385887',)


,Persona,Profesión,Personaje
0,Edward Norton,actor,"[""Lionel Essrog""]"
1,Gugu Mbatha-Raw,actor,"[""Laura Rose""]"
2,Alec Baldwin,actor,"[""Moses Randolph""]"
3,Willem Dafoe,actor,"[""Paul Randolph""]"
4,Bruce Willis,actor,"[""Frank Minna""]"
5,Ethan Suplee,actor,"[""Gilbert Coney""]"
6,Cherry Jones,actor,"[""Gabby Horowitz""]"
7,Bobby Cannavale,actor,"[""Tony Vermonte""]"
8,Dallas Roberts,actor,"[""Danny Fantl""]"
9,Josh Pais,actor,"[""William Lieberman""]"


Cantidad de títulos por tipo

In [35]:
session.execute(select(Titulo.tipo, func.count()).group_by(Titulo.tipo)).all()

2025-11-02 22:44:18,263 INFO sqlalchemy.engine.Engine SELECT titulo.tipo, count(*) AS count_1 
FROM titulo GROUP BY titulo.tipo
2025-11-02 22:44:18,265 INFO sqlalchemy.engine.Engine [cached since 603.9s ago] ()


[(<TipoTitulo.PELICULA: 'movie'>, 10052),
 (<TipoTitulo.TV_MOVIE: 'tvMovie'>, 1202)]

Cantidad de títulos por género

In [36]:
session.execute(
    select(Genero.nombre, func.count(Titulo_Genero.id_titulo))
    .join(Titulo_Genero)
    .group_by(Genero.nombre)
).all()

2025-11-02 22:44:29,199 INFO sqlalchemy.engine.Engine SELECT genero.nombre, count(titulo_genero.id_titulo) AS count_1 
FROM genero JOIN titulo_genero ON genero.id = titulo_genero.id_genero GROUP BY genero.nombre
2025-11-02 22:44:29,201 INFO sqlalchemy.engine.Engine [cached since 601s ago] ()


[('Action', 1124),
 ('Adventure', 642),
 ('Animation', 467),
 ('Biography', 723),
 ('Comedy', 2877),
 ('Crime', 919),
 ('Documentary', 5900),
 ('Drama', 5902),
 ('Family', 580),
 ('Fantasy', 413),
 ('Game-Show', 1),
 ('History', 562),
 ('Horror', 1199),
 ('Music', 510),
 ('Musical', 164),
 ('Mystery', 642),
 ('News', 11),
 ('Reality-TV', 37),
 ('Romance', 1136),
 ('Sci-Fi', 330),
 ('Sport', 289),
 ('Talk-Show', 11),
 ('Thriller', 1326),
 ('War', 149),
 ('Western', 43)]

búsqueda por nombre alternativo parcial

In [38]:
busqueda = "Retrato de una mujer en llamas"

resultados = session.execute(
    select(Titulo.id, Titulo.titulo, Titulo_Alternativo.titulo, Titulo_Alternativo.region, Titulo_Alternativo.idioma)
    .join(Titulo_Alternativo, Titulo.id == Titulo_Alternativo.id_titulo)
    .where(Titulo_Alternativo.titulo.ilike(f"%{busqueda}%"))
).all()

df_akas = pd.DataFrame(resultados, columns=["ID", "Título Original", "AKA", "Región", "Idioma"])
display(df_akas if not df_akas.empty else "❌ No se encontraron coincidencias.")

2025-11-02 22:46:21,604 INFO sqlalchemy.engine.Engine SELECT titulo.id, titulo.titulo, titulo_alternativo.titulo AS titulo_1, titulo_alternativo.region, titulo_alternativo.idioma 
FROM titulo JOIN titulo_alternativo ON titulo.id = titulo_alternativo.id_titulo 
WHERE lower(titulo_alternativo.titulo) LIKE lower(?)
2025-11-02 22:46:21,605 INFO sqlalchemy.engine.Engine [cached since 90.42s ago] ('%Retrato de una mujer en llamas%',)


,ID,Título Original,AKA,Región,Idioma
0,tt8613070,Portrait of a Lady on Fire,Retrato de una mujer en llamas,US,es


In [49]:
resultados = session.execute(
  select(Persona.nombre, Profesion.nombre, Titulo.titulo)
  .join(Profesion_Titulo, Persona.id == Profesion_Titulo.id_persona)
  .join(Profesion, Profesion.id == Profesion_Titulo.id_profesion)
  .join(Titulo, Titulo.id == Profesion_Titulo.id_titulo)
  .limit(10)
).all()

df_profesiones = pd.DataFrame(resultados, columns=["Persona", "Profesión", "Título"])
display(df_profesiones)

2025-11-02 23:02:08,591 INFO sqlalchemy.engine.Engine SELECT persona.nombre, profesion.nombre AS nombre_1, titulo.titulo 
FROM persona JOIN profesion_titulo ON persona.id = profesion_titulo.id_persona JOIN profesion ON profesion.id = profesion_titulo.id_profesion JOIN titulo ON titulo.id = profesion_titulo.id_titulo
 LIMIT ? OFFSET ?
2025-11-02 23:02:08,592 INFO sqlalchemy.engine.Engine [generated in 0.00242s] (10, 0)


,Persona,Profesión,Título
0,Geraldine O'Rawe,actor,Mariette in Ecstasy
1,Eva Marie Saint,actor,Mariette in Ecstasy
2,Alex Appel,actor,Mariette in Ecstasy
3,Nancy Beatty,actor,Mariette in Ecstasy
4,Rutger Hauer,actor,Mariette in Ecstasy
5,John Mahoney,actor,Mariette in Ecstasy
6,Mary McDonnell,actor,Mariette in Ecstasy
7,Cara Pifko,actor,Mariette in Ecstasy
8,Leigh Taylor-Young,actor,Mariette in Ecstasy
9,Megan Banning,actor,Mariette in Ecstasy


In [47]:
resultados = session.execute(
  select(Titulo.titulo, Puntaje.promedio, Puntaje.cantidad_votos)
  .join(Puntaje, Titulo.id == Puntaje.id_titulo)
  .order_by(Puntaje.cantidad_votos.desc())
  .limit(10)
).all()

df_puntajes = pd.DataFrame(resultados, columns=["Título", "Rating", "Votos"])
display(df_puntajes)

2025-11-02 23:00:30,931 INFO sqlalchemy.engine.Engine SELECT titulo.titulo, puntaje.promedio, puntaje.cantidad_votos 
FROM titulo JOIN puntaje ON titulo.id = puntaje.id_titulo ORDER BY puntaje.cantidad_votos DESC
 LIMIT ? OFFSET ?
2025-11-02 23:00:30,937 INFO sqlalchemy.engine.Engine [generated in 0.00591s] (10, 0)


,Título,Rating,Votos
0,Joker,8.3,1635878
1,Avengers: Endgame,8.4,1384271
2,Parasite,8.5,1094522
3,Once Upon a Time... in Hollywood,7.6,925190
4,Knives Out,7.9,828929
5,1917,8.2,741334
6,Captain Marvel,6.7,642205
7,Spider-Man: Far from Home,7.4,611286
8,Ford v Ferrari,8.1,539592
9,Star Wars: Episode IX - The Rise of Skywalker,6.4,525995


In [43]:
resultados = session.execute(
  select(
    Titulo.titulo,
    Genero.nombre,
    Persona.nombre,
    Profesion.nombre,
    Puntaje.promedio
  )
  .join(Titulo_Genero, Titulo.id == Titulo_Genero.id_titulo, isouter=True)
  .join(Genero, Genero.id == Titulo_Genero.id_genero, isouter=True)
  .join(Profesion_Titulo, Titulo.id == Profesion_Titulo.id_titulo, isouter=True)
  .join(Persona, Persona.id == Profesion_Titulo.id_persona, isouter=True)
  .join(Profesion, Profesion.id == Profesion_Titulo.id_profesion, isouter=True)
  .join(Puntaje, Titulo.id == Puntaje.id_titulo, isouter=True)
  .limit(20)
).all()

df_integridad = pd.DataFrame(resultados, columns=["Título", "Género", "Persona", "Profesión", "Rating"])
display(df_integridad)

2025-11-02 22:58:44,212 INFO sqlalchemy.engine.Engine SELECT titulo.titulo, genero.nombre, persona.nombre AS nombre_1, profesion.nombre AS nombre_2, puntaje.promedio 
FROM titulo LEFT OUTER JOIN titulo_genero ON titulo.id = titulo_genero.id_titulo LEFT OUTER JOIN genero ON genero.id = titulo_genero.id_genero LEFT OUTER JOIN profesion_titulo ON titulo.id = profesion_titulo.id_titulo LEFT OUTER JOIN persona ON persona.id = profesion_titulo.id_persona LEFT OUTER JOIN profesion ON profesion.id = profesion_titulo.id_profesion LEFT OUTER JOIN puntaje ON titulo.id = puntaje.id_titulo
 LIMIT ? OFFSET ?
2025-11-02 22:58:44,212 INFO sqlalchemy.engine.Engine [generated in 0.00223s] (20, 0)


,Título,Género,Persona,Profesión,Rating
0,Mariette in Ecstasy,Drama,Rutger Hauer,actor,7.0
1,Mariette in Ecstasy,Drama,John Mahoney,actor,7.0
2,Mariette in Ecstasy,Drama,Mary McDonnell,actor,7.0
3,Mariette in Ecstasy,Drama,Eva Marie Saint,actor,7.0
4,Mariette in Ecstasy,Drama,John Bailey,director,7.0
5,Mariette in Ecstasy,Drama,Alex Appel,actor,7.0
6,Mariette in Ecstasy,Drama,Megan Banning,actor,7.0
7,Mariette in Ecstasy,Drama,Nancy Beatty,actor,7.0
8,Mariette in Ecstasy,Drama,Ron Hansen,writer,7.0
9,Mariette in Ecstasy,Drama,Christopher Klatman,composer,7.0


In [46]:
tablas = ["titulo", "genero", "titulo_genero", "persona", "profesion", 
          "profesion_titulo", "puntaje", "titulo_alternativo"]

for tabla in tablas:
  total = session.execute(text(f"SELECT COUNT(*) FROM {tabla}")).scalar()
  print(f"{tabla:<20} → {total:,} registros")

2025-11-02 23:00:03,313 INFO sqlalchemy.engine.Engine SELECT COUNT(*) FROM titulo
2025-11-02 23:00:03,319 INFO sqlalchemy.engine.Engine [generated in 0.00726s] ()
titulo               → 11,254 registros
2025-11-02 23:00:03,329 INFO sqlalchemy.engine.Engine SELECT COUNT(*) FROM genero
2025-11-02 23:00:03,330 INFO sqlalchemy.engine.Engine [generated in 0.00137s] ()
genero               → 25 registros
2025-11-02 23:00:03,337 INFO sqlalchemy.engine.Engine SELECT COUNT(*) FROM titulo_genero
2025-11-02 23:00:03,340 INFO sqlalchemy.engine.Engine [generated in 0.00385s] ()
titulo_genero        → 25,957 registros
2025-11-02 23:00:03,346 INFO sqlalchemy.engine.Engine SELECT COUNT(*) FROM persona
2025-11-02 23:00:03,347 INFO sqlalchemy.engine.Engine [generated in 0.00136s] ()
persona              → 133,957 registros
2025-11-02 23:00:03,363 INFO sqlalchemy.engine.Engine SELECT COUNT(*) FROM profesion
2025-11-02 23:00:03,369 INFO sqlalchemy.engine.Engine [generated in 0.00679s] ()
profesion        

In [50]:
session.close()
print("Sesión cerrada correctamente.")

2025-11-02 23:02:50,177 INFO sqlalchemy.engine.Engine ROLLBACK
Sesión cerrada correctamente.
